# 11. MLflow - 실험 추적

PDF `MLflow.pdf` 11~28쪽 실습

**준비물**: 터미널을 하나 열어 Tracking 서버를 켜 두고 실습한다.

```powershell
cd C:\Users\samsung1\git_project\samsung_practice\머신러닝분류\mlflow_practice
C:\Users\samsung1\git_project\samsung_practice\.venv\Scripts\mlflow.exe server --host 127.0.0.1 --port 5000 --backend-store-uri sqlite:///mlflow.db --serve-artifacts --artifacts-destination ./artifacts
```

- `cd`로 `mlflow_practice` 폴더에 들어가야 기존 기록(`mlflow.db`)이 그대로 보인다.
- 커널은 `samsung_practice (.venv)`를 사용한다.
- UI 주소: http://127.0.0.1:5000

## 1. 설치 확인 (11~12쪽)

In [2]:
import mlflow

print(mlflow.__version__)

3.16.1


## 2. 어디에 기록할지 정하기 (14쪽)

- `set_tracking_uri` : 기록을 어디로 보낼지 정한다. 켜 둔 서버를 가리킨다.
- `set_experiment` : 실험 묶음(폴더, 실험) 이름. 없으면 새로 만들어진다.

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("breast-cancer-basic-tracking")
#   - 실험 = Run 들을 담는 폴더. 이후 모든 start_run()이 여기에 들어간다.
#   - 만약 서버가 꺼져 있으면 여기서 멈춘다. 실험이 있는지 서버에 물어봐야 하기 때문

print(mlflow.get_tracking_uri())
#   - http://127.0.0.1:5000 이 찍히면 정상 출력.


2026/09/23 09:12:05 INFO mlflow.tracking.fluent: Experiment with name 'breast-cancer-basic-tracking' does not exist. Creating a new experiment.


http://127.0.0.1:5000


## 3. 데이터 준비 (18쪽)

유방암 진단 데이터셋을 학습용/테스트용으로 나눈다.

In [4]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape)

(455, 30) (114, 30)


## 4. 첫 Run - 파라미터와 지표 기록 (15, 16, 19쪽)

`with mlflow.start_run():` 블록 하나가 **Run 한 개**다.

- `log_param` : 학습 조건 (n_estimators, max_depth)
- `log_metric` : 결과 점수 (accuracy)

In [5]:
with mlflow.start_run(run_name="rf-n50-d3"):
    model = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)
    model.fit(X_train, y_train)

    acc = accuracy_score(y_test, model.predict(X_test))

    mlflow.log_params({"n_estimators": 50, "max_depth": 3})
    mlflow.log_metric("accuracy", acc)

print("accuracy =", acc)

🏃 View run rf-n50-d3 at: http://127.0.0.1:5000/#/experiments/2/runs/f7f649046b1147819dd2bce449cda3db
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
accuracy = 0.9649122807017544


> UI(http://127.0.0.1:5000)에서 `breast-cancer-basic-tracking` 실험을 열어 Run이 생겼는지 확인한다.

## 5. Tag 남기기 (20쪽)

검색과 설명에 쓰는 문자열 메모. Run 안에서 기록한다.

In [ ]:
with mlflow.start_run(run_name="rf-with-tags"):
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))

#   - with 블록 하나 = Run 하나
#   - 블록에 들어가면 Run 시작, 빠져나오면 자동으로 종료 처리된다.
#   - run_name은 UI에 보이는 이름일 뿐 고유 식별자는 아니다.
#   - 똑같은 이름을 여러 번 써도 각각 다른 Run이 된다.
#   - 진짜 식별자는 run_id이다. (자동 생성되는 32자리 문자열)

    mlflow.log_params({"n_estimators": 100, "max_depth": 5})
#   - log_params는 "장부에 기록하는" 것일 뿐 모델을 바꾸지 않는다.
#   - 이 코드를 빼면 UI에서 설정값을 못 알아본다.
#   - 실제로 max_depth가 5인데, 기록 코드에서 max_depth를 10으로 적으면 UI에선 10으로 보임

    mlflow.log_metric("accuracy", acc)
#   - 지표 기록: 성능 숫자, 정렬 / 비교 / 그래프가 되는 값

    mlflow.set_tag("model_type", "RandomForest")
    mlflow.set_tag("dataset", "breast_cancer")
    mlflow.set_tag("owner", "student")
#   - 태그: 검색하고 분류하기 위한 문자열 메모로, 파라미터와 달리 나중에 덮어쓸(수정할) 수 있다.

print("accuracy =", acc)

🏃 View run rf-with-tags at: http://127.0.0.1:5000/#/experiments/2/runs/0303878a63c84070a7ace76628aff3ee
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
accuracy = 0.9649122807017544


## 6. Artifact - 파일 남기기 (17쪽)

그림, CSV, JSON처럼 '파일 그 자체'를 남길 때 쓴다.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

with mlflow.start_run(run_name="rf-artifact"):
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    mlflow.log_metric("accuracy", accuracy_score(y_test, pred))

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax)
#   - 정답과 예측을 주면 혼동행렬을 그려 주는 코드이다.
#   - ax=ax

    mlflow.log_figure(fig, "plots/confusion_matrix.png")
#   - figure 객체를 그대로 서버에 올린다. 로컬에 savefig로 저장할 필요가 없다.
#   - 두 번째 인자는 Artifact 안에서의 경로로, 슬래시(/)를 쓰면 UI에 폴더로 나뉘어 보인다.

    plt.close(fig)
#   - close()를 통해 닫아 주는 이유
#       1) 안 닫으면 노트북 셀 출력에 그림이 한 번 더 뜬다.
#       2) 반복 실행하면 figure가 메모리에 계속 쌓인다.(20개 넘으면 matplotlib 경고 뜸)

print("저장 완료 - UI의 Artifacts 탭에서 확인")

🏃 View run rf-artifact at: http://127.0.0.1:5000/#/experiments/2/runs/80033ae8fab84f37a7e64718b6d9252d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
저장 완료 - UI의 Artifacts 탭에서 확인


## 7. Autolog - 한 줄로 자동 기록 (21쪽)

`log_param`, `log_metric`을 직접 부르지 않아도 사이킷런이 알아서 기록한다.

단, 자동으로 남는 건 **학습(training) 점수**다. 검증 점수는 따로 남겨야 한다. (73쪽)

In [8]:
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="autolog-RandomForest"):
    model = RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)
    model.fit(X_train, y_train)

print("자동 기록 완료")

2026/09/23 09:14:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\samsung1\git_project\samsung_practice\.venv\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


🏃 View run autolog-RandomForest at: http://127.0.0.1:5000/#/experiments/2/runs/d703c18ba5f14c4f9d053daad7f80f2e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
자동 기록 완료


## 8. 실습 - Autolog로 여러 모델 비교 (22쪽)

모델만 바꿔가며 돌려도 각 모델에 맞는 파라미터가 알아서 기록된다.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

models = {
    "RandomForest": RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=2000),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
}

for name, model in models.items():
    with mlflow.start_run(run_name=f"autolog-{name}"):
        model.fit(X_train, y_train)
        mlflow.log_metric("test_accuracy", model.score(X_test, y_test))

mlflow.sklearn.autolog(disable=True)  # 수동 기록으로 돌아갈 때는 꼭 끈다
print("3개 모델 기록 완료")

2026/09/23 09:14:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\samsung1\git_project\samsung_practice\.venv\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


🏃 View run autolog-RandomForest at: http://127.0.0.1:5000/#/experiments/2/runs/719575ff5f6b46c3948b13e290801cee
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


c:\Users\samsung1\git_project\samsung_practice\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/09/23 09:14:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/mode

🏃 View run autolog-LogisticRegression at: http://127.0.0.1:5000/#/experiments/2/runs/672315a04c60475b89c4727363a2710a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/09/23 09:14:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\samsung1\git_project\samsung_practice\.venv\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


🏃 View run autolog-GradientBoosting at: http://127.0.0.1:5000/#/experiments/2/runs/48157272fed24c6a91f950dc948b5cd5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
3개 모델 기록 완료


## 9. UI에서 비교하기 (24~28쪽)

http://127.0.0.1:5000 에서

1. 왼쪽 메뉴 **Model training** 선택
2. `breast-cancer-basic-tracking` 실험 클릭
3. 검색창에 조건 입력해 보기 -> `metrics.accuracy > 0.95`
4. Run 여러 개 체크 -> **Compare** 버튼 -> Parallel Coordinates Plot 확인

## 10. 코드로 조회하기 (67쪽 맛보기)

`search_runs`는 결과를 pandas DataFrame으로 돌려준다.

In [10]:
runs = mlflow.search_runs(order_by=["metrics.accuracy DESC"])
runs[["run_id", "tags.mlflow.runName", "metrics.accuracy"]].head(10)

,run_id,tags.mlflow.runName,metrics.accuracy
0,80033ae8fab84f37a7e64718b6d9252d,rf-artifact,0.964912
1,0303878a63c84070a7ace76628aff3ee,rf-with-tags,0.964912
2,f7f649046b1147819dd2bce449cda3db,rf-n50-d3,0.964912
3,48157272fed24c6a91f950dc948b5cd5,autolog-GradientBoosting,NaN
4,672315a04c60475b89c4727363a2710a,autolog-LogisticRegression,NaN
5,719575ff5f6b46c3948b13e290801cee,autolog-RandomForest,NaN
6,d703c18ba5f14c4f9d053daad7f80f2e,autolog-RandomForest,NaN
